cuEclat

A short tutorial for cuEclat. See the [algorithm guide](https://github.com/udayRage/cuPAMI/blob/main/docs/algorithms/cuEclat.md) for details.

In Colab, choose Runtime → Change runtime type → GPU. Install the library from GitHub and a CuPy wheel matching your CUDA installation. The setup below uses CUDA 12. Run setup once; skip it if your local environment is already configured.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cupami") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/udayRage/cuPAMI.git"])
if importlib.util.find_spec("cupy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cupy-cuda12x"])

In [2]:
import cupy as cp

print("CUDA devices:", cp.cuda.runtime.getDeviceCount())


CUDA devices: 2


Create a small dataset

Use BinaryTransactionalDatabase with a fixed seed so the input can be reproduced.

In [3]:
from cupami import BinaryTransactionalDatabase, cuEclat

generator = BinaryTransactionalDatabase(transactions=100, items=8, avg_length=3, seed=42)
data = generator.create()
print(data[:5])


[[2, 5, 6, 7], [1, 2, 4, 5], [1], [2, 4, 7], [1, 2, 7]]


Mine patterns

`minSup=0.1` means 10% of transactions.

In [4]:
miner = cuEclat(data, minSup=0.1)
miner.mine()


View results

Show the first ten patterns and their supports.

In [5]:
from itertools import islice

patterns = miner.getPatterns()
print("Patterns:", len(patterns))
for pattern, support in islice(patterns.items(), 10):
    print(pattern, support)


Patterns: 35
1 44
2 45
3 34
4 29
5 39
6 48
7 36
8 37
1	2 22
1	3 18


Timing and output

Times are in seconds. Save the patterns to a local text file.

In [6]:
print("Loading:", miner.getLoadingTime())
print("Mining:", miner.getMiningTime())
print("Total:", miner.getTotalRuntime())
miner.save("patterns.txt")


Loading: 0.00010592001490294933
Mining: 0.16058643301948905
Total: 0.160692353034392
